PART 03：四个新工具逐个拆——每个都短，但每个都有讲究
插座板装好了，该造电器了。四个新工具，一个一个拆，每个都附一段值得停下来想的设计点。

read_file：把"看多少"的控制权交给模型

In [ ]:
def run_read(path: str, limit: int | None = None) -> str:
    try:
        lines = safe_path(path).read_text(encoding="utf-8").splitlines()
        if limit and limit < len(lines):
            lines = lines[:limit] + [f"... ({len(lines) - limit} more lines)"]
        return "\n".join(lines)
    except Exception as e:
        return f"Error: {e}"

核心就一行 read_text，讲究在 limit 参数：模型可以只要前 50 行。超出的部分不硬砍，而是补一行 ... (3200 more lines)——截断要留痕，让模型知道文件还有多少没看，想看随时再来。

回忆一下上篇 bash 执行器里那个"输出截断到 50000 字符"的创可贴：那是工具替模型做主、一刀切到固定长度。现在 limit 把裁量权还给了模型自己——同一个问题的两种解法，从"被动保险"进化成"主动可控"。

write_file：PART 01 那些坑的总清算

In [ ]:
def run_write(path: str, content: str) -> str:
    try:
        file_path = safe_path(path)
        file_path.parent.mkdir(parents=True, exist_ok=True)
        file_path.write_text(content, encoding="utf-8")
        return f"Wrote {len(content)} bytes to {path}"
    except Exception as e:
        return f"Error: {e}"

看着平平无奇对吧？全部的威力在于 content 是一个结构化参数，而不是 shell 字符串。

模型调这个工具时，想写什么就原样放进 JSON 的 content 字段——带单引号？带双引号？带 $ 符号？带一百个换行？统统原样落盘，从头到尾没有一个字符需要转义。PART 01 里那场引号嵌套的噩梦，在这套机制下根本没有发生的土壤，因为 shell 根本不在场。

mkdir(parents=True) 也顺手把"目录还不存在"这种琐碎失败消掉了——写 src/utils/helpers.py 时自动建好两级目录。工具的返回信息也值得学：Wrote 523 bytes to config.py，成功也要给回执，写入了多少字节、写到哪，模型下一圈心里有数。

edit_file：精确替换一次，找不到就明说

In [ ]:
def run_edit(path: str, old_text: str, new_text: str) -> str:
    try:
        file_path = safe_path(path)
        text = file_path.read_text(encoding="utf-8")
        if old_text not in text:
            return f"Error: text not found in {path}"
        file_path.write_text(text.replace(old_text, new_text, 1), encoding="utf-8")
        return f"Edited {path}"
    except Exception as e:
        return f"Error: {e}"

改文件的思路不是 sed 正则，而是最笨也最稳的：给原文，给新文，精确替换第一处。

为什么"笨"反而是对的？想一下 sed 的方案：模型得生成一段正则，正则本身又是一层"翻译"，转义翻车的故事在 regex 上重演一遍；而且正则是模糊匹配，.* 手一滑，误伤范围根本不可控。

精确文本替换没有这个问题：old_text 在文件里逐字找，找到才改，找不到立刻报 "Error: text not found"。注意这个报错的去向——它不是抛给用户的，是塞回给模型的。模型收到"原文没找到"，下一圈自然会先 read_file 看一眼文件现状，再修正 old_text 重试。改错了（比如找到了两处相似文本）也只动第一处，replace(..., 1) 的那个 1 就是爆炸半径上限。

一个"精确匹配、失败即反馈、爆炸半径有限"的编辑原语，比一个功能强大但行为含糊的正则机器，对 Agent 友好得多。给模型用的工具，确定性永远优先于表现力。

glob：找文件，顺便管住自己的嘴

In [ ]:
def run_glob(pattern: str) -> str:
    import glob as g
    try:
        matches = sorted({
            match for match in g.glob(pattern, root_dir=WORKDIR, recursive=True)
            if (WORKDIR / match).resolve().is_relative_to(WORKDIR)
        })
        shown = matches[:200]
        if len(matches) > 200:
            shown.append("... (more matches omitted; narrow the pattern)")
        return "\n".join(shown) if shown else "(no matches)"
    except Exception as e:
        return f"Error: {e}"

模式匹配找文件，**/*.py 递归捞出全部 Python 文件。两个细节：结果排序去重，保证同样的问题每次得到同样的答案——工具输出越确定，模型行为越稳定；超过 200 条就截断并提示"收窄你的 pattern"——又是"截断留痕"，跟 read_file 一个家风。

safe_path：一道只围了四分之三的围栏
你可能注意到上面每个函数的第一步都是 safe_path(path)，它是什么：

In [ ]:
def safe_path(p: str) -> Path:
    path = (WORKDIR / p).resolve()
    if not path.is_relative_to(WORKDIR):
        raise ValueError(f"Path escapes workspace: {p}")
    return path

三步：把相对路径接到工作目录下、resolve() 展开所有 .. 和软链、然后检查结果还圈在工作区里吗。模型想调 read_file("../../etc/passwd")？.. 被 resolve 展开后路径落在工作区外面，直接抛错。绕、编码、层层跳转，都逃不过 resolve 这一关——不看你说去哪，只看你实际到了哪。

但我要指着这个设计里的一个洞，大声念三遍：

safe_path 只保护四个文件工具，bash 完全不设防。

模型想读工作区外的文件？read_file 被拦。但它转头调 bash 跑一句 cat /etc/passwd，畅通无阻；rm -rf ~/重要目录 同样拦不住——上篇那个粗糙黑名单，依然是目前唯一的防线。

这不是疏忽，是节奏：先把"工具分发"这个机制立起来，权限系统留到下一篇专门拆。但你必须现在就知道这个洞在哪，每个工具各自为政的安全检查，挡不住那条绕开所有检查的通用通道。这是安全设计里反复出现的教训，记住它，下一篇看权限系统时会更有感觉。

PART 04：跑起来——看模型怎么用新工具
代码拼完，跑：

代码拼完，跑：

python code.py
四个实录，重点看模型行为的细节。

实录一：读文件，它不再绕 bash 了
读一下 README.md 前 30 行,讲讲这个项目是干嘛的
终端输出：

> read_file
(前 30 行内容)
This project is a minimal Claude Code clone...
模型直接调了 read_file，参数 {"path": "README.md", "limit": 30}——没有 cat，没有 head，没有管道。更妙的是那个 limit: 30：你只说"前 30 行"，它自己就把这个数填进了参数。意图直达动作，一步到位。

实录二：还债时刻——那段刁钻内容，一发入魂
把 PART 01 里翻车的那段内容原样再要一次：

创建 config.py,内容是:MSG = "It's a $test",再创建一个 10 行的 demo.py,
里面每行都打印一句话,两个文件都写完
模型这轮调了两次 write_file，content 参数里原封不动地躺着单引号、双引号、$ 符号、十行换行——落盘的文件和它的意图逐字节一致。上一版引号嵌套的鬼画符，这一版根本没有出场机会。

跑一下 python config.py，输出 It's a $test，完整无缺。债，清了。

实录三：一轮多工具——它真的会一拳打出三张牌
这是本篇最有观察价值的一刻。输入：

读 README.md 和 requirements.txt,再列出目录下所有 py 文件
盯着终端，模型这一轮同时发出了三个申请：

> read_file    (README.md)
> read_file    (requirements.txt)
> glob         (**/*.py)
一轮回复里塞了三个 tool_use 块。上一篇欠的第二个问题有了答案：会，一轮多工具是模型的原生行为，不用你做任何特殊支持——只要 API 响应里出现多个 tool_use 块，你的循环自然就逐个处理了。

那第三个问题——会不会互相踩？也不会，原因简单得出人意料：

for block in tool_calls:
    handler = TOOL_HANDLERS.get(block.name)
    output = handler(**block.input)
这个 for 是串行的：第一个 read_file 执行完、拿到结果，才轮到第二个，最后才是 glob。全程一条车道，没有任何两个工具同时碰文件系统，没有并发，就没有竞态。三个结果各自带着自己的 tool_use_id，像三张不同编号的回执，配对塞回账本，下一圈模型一次性读到三份结果，合并推理。

当然，串行的代价是慢——三个互不依赖的读取，本可以同时跑。真实 Claude Code 后来确实把互不依赖的工具调用改成并发执行，那是"提速"的需求倒逼出来的复杂度；而我们的原则不变：先把正确的跑通，再让快的起飞。初学者直接上并发，九成时间在调锁的 bug，不在理解机制。

实录四：分寸感——五个工具之间的选择
多玩几轮，你会看到模型在五个工具之间自如切换：读文件用 read_file、改文件用 edit_file、找文件用 glob，但"删掉 test 目录"它转身就抄 bash（rm -rf test/）——因为工具箱里没有"删除"这个原语，bash 就是那个万能的兜底。

专用工具管常用高频路径，bash 兜住长尾。这个组合的分工感，跟 Claude Code 本尊如出一辙：它也有自己的一整套专用读写编辑工具，但遇到没覆盖的场景，从不犹豫切 bash。

最后照例一句严肃提醒：bash 依旧不设防，safe_path 围栏管不到它。继续在临时目录里玩，别拿有重要文件的目录做实验田。